## 랭그래프 실습

1. 질문 : 체력이 안 좋고, 살이 계속 찌는데 어떤 운동을 할까?
2. 추출에이전트 : 사용자의 질문에서 증상을 추출
3. 후보에이전트 : 증상을 해결할 수 있는 운동리스트 추출
4. 답변생성에이전트 : 증상과 운동리스트를 개조식으로 출력하도록 답변 생성

In [1]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. 상태 정의
class AgentState(TypedDict):
    query: str           # 사용자 질문
    symptoms: str        # 추출된 증상
    exercise_list: str   # 추천 운동 리스트
    result: str          # 최종 응답

In [2]:
# 2. LLM 초기화
llm = Ollama(model="exaone3.5:2.4b")

/var/folders/xp/3lw4xc2935531gg4z132x0780000gn/T/ipykernel_71775/2031171905.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="exaone3.5:2.4b")


In [3]:
# 3. 에이전트 정의
# 추출 에이전트 : 질문에서 증상 추출
extractor_prompt = PromptTemplate.from_template("""
                                                사용자의 질문에서 신체적 증상이나 고민에 해당하는 단어 또는 구를 추출하세요.
                                                결과는 쉼표로 구분된 문자열로 출력하세요.
                                                질문: {query}
                                                """)

def extractor_agent(state: AgentState):
    chain = extractor_prompt | llm
    symptoms = chain.invoke({"query": state["query"]})
    return {**state, "symptoms": symptoms.strip()}

# 후보 에이전트 : 증상에 맞는 운동 리스트 추출
exercise_prompt = PromptTemplate.from_template("""
                                                다음 증상 또는 고민을 해결하는 데 도움이 되는 운동을 5가지 추천하세요.
                                                운동 이름만 쉼표로 구분하여 출력하세요.
                                                증상: {symptoms}
                        """)

def exercise_agent(state: AgentState):
    chain = exercise_prompt | llm
    exercise_list = chain.invoke({"symptoms": state["symptoms"]})
    return {**state, "exercise_list": exercise_list.strip()}



# 답변생성 에이전트 : 증상과 운동리스트를 개조식으로 출력
answer_prompt = PromptTemplate.from_template("""
사용자의 증상과 추천 운동 목록을 아래 형식(개조식)으로 정리하여 출력하세요.

[증상]
- 증상1
- 증상2

[추천 운동]
1. 운동명 : 이 운동이 도움이 되는 이유 한 문장

증상: {symptoms}
추천 운동 목록: {exercise_list}
""")


# 최종 답변 함수
def answer_agent(state: AgentState):
    chain = answer_prompt | llm
    answer = chain.invoke({
        "symptoms": state["symptoms"],
        "exercise_list": state["exercise_list"]
    })
    return {**state, "result": answer.strip()}

In [4]:
# 4. LangGraph 정의
graph = StateGraph(AgentState)
graph.add_node("extractor", extractor_agent)
graph.add_node("exercise", exercise_agent)
graph.add_node("answer", answer_agent)

graph.set_entry_point("extractor")
graph.add_edge("extractor", "exercise")
graph.add_edge("exercise", "answer")
graph.add_edge("answer", END)

app = graph.compile()

In [10]:
# 5. 구조 확인
print("<LangGraph 구조>")
app.get_graph().print_ascii()

<LangGraph 구조>
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| extractor |  
+-----------+  
      *        
      *        
      *        
+----------+   
| exercise |   
+----------+   
      *        
      *        
      *        
  +--------+   
  | answer |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [14]:
# 6. 실행
query = "체력이 안 좋고, 살이 계속 찌는데 어떤 운동을 할까?"
result = app.invoke({"query": query})

print("<최종 응답>")
print(result["result"])

<최종 응답>
## 사용자 증상 및 추천 운동 목록

**[증상]**
- 체력 저하 및 체지방 감소 어려움
- 체중 관리 필요성 증가

**[추천 운동]**
1. **줄넘기** : 심장 건강 증진 및 칼로리 소모 효과가 뛰어나 체지방 감소에 효과적입니다. 빠른 템포의 줄넘기는 신체 활동량을 늘려 에너지 소비를 촉진합니다.
2. **스쿼트 루틴** : 근력 강화 및 신체 균형 향상에 도움을 주며, 하체 근육량 증가로 인해 기초대사량이 높아져 자연스럽게 체중 감량에 기여합니다.
3. **버피 테스트** : 전신 운동으로 심폐기능 강화와 함께 근육량 증가를 유도하여 체지방 감소와 함께 근력 향상에 효과적입니다.
4. **등산 또는 자전거 타기** : 유산소 운동으로 지구력 향상과 함께 꾸준한 활동을 통해 체중 관리에 도움을 줍니다. 거리와 속도 조절을 통해 개인의 체력 수준에 맞춰 운동 강도를 조절할 수 있습니다.
5. **요가** : 유연성 향상과 스트레스 해소에 효과적입니다. 규칙적인 요가는 기초 대사량 증진과 균형 잡힌 체중 관리에 도움을 줄 수 있습니다. 특히, 코어 근육 강화를 통해 자세 개선과 체지방 관리에 효과적입니다.
